In [1]:
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
import numpy as np
import re
import html
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder




In [ ]:

train_rows = 0

for chunk in pd.read_csv("Train.csv", chunksize=50000):
    train_rows += len(chunk)

print("Total train rows:", train_rows)

Total train rows: 6034195


In [5]:
train_rows = 0

for chunk in pd.read_csv("Test.csv", chunksize=50000):
    train_rows += len(chunk)

print("Total train rows:", train_rows)

Total train rows: 2013337


In [1]:
import pandas as pd

train_sample = pd.read_csv("Train.csv", nrows=200000)
test_sample = pd.read_csv("Test.csv", nrows=100000)

print(train_sample.shape)
print(test_sample.shape)

(200000, 4)
(100000, 3)


# tag selection + filtering validation

In [ ]:
from collections import Counter

tag_counter = Counter()
for chunk in pd.read_csv("Train.csv", chunksize=50000):
    for tags in chunk["Tags"].dropna():
        tag_counter.update(str(tags).split())
print(tag_counter.most_common(50))


[('c#', 463526), ('java', 412189), ('php', 392451), ('javascript', 365623), ('android', 320622), ('jquery', 305614), ('c++', 199280), ('python', 184928), ('iphone', 183573), ('asp.net', 177334), ('mysql', 172182), ('html', 165507), ('.net', 162359), ('ios', 136080), ('objective-c', 133932), ('sql', 132465), ('css', 129107), ('linux', 127606), ('ruby-on-rails', 116883), ('windows', 98100), ('c', 95453), ('sql-server', 74921), ('ruby', 73502), ('wpf', 65836), ('xml', 64157), ('ajax', 62239), ('database', 59799), ('regex', 59223), ('windows-7', 58487), ('asp.net-mvc', 57859), ('xcode', 52513), ('django', 52030), ('osx', 51812), ('arrays', 50055), ('vb.net', 46653), ('eclipse', 44092), ('json', 43451), ('facebook', 43393), ('ruby-on-rails-3', 43166), ('ubuntu', 43002), ('performance', 39377), ('networking', 38323), ('string', 37802), ('multithreading', 37619), ('winforms', 37370), ('security', 34499), ('visual-studio-2010', 34433), ('asp.net-mvc-3', 34422), ('bash', 33116), ('homework', 32

In [ ]:
top_400_tags = tag_counter.most_common(400)

In [ ]:
top_400_df = pd.DataFrame(top_400_tags, columns=["tag", "count"])
top_400_df.index = range(1, len(top_400_df) + 1)

In [ ]:
parts = np.array_split(top_400_df, 16)

In [44]:
part_number = 1
display(parts[part_number - 1])

array([['c#', 463526],
       ['java', 412189],
       ['php', 392451],
       ['javascript', 365623],
       ['android', 320622],
       ['jquery', 305614],
       ['c++', 199280],
       ['python', 184928],
       ['iphone', 183573],
       ['asp.net', 177334],
       ['mysql', 172182],
       ['html', 165507],
       ['.net', 162359],
       ['ios', 136080],
       ['objective-c', 133932],
       ['sql', 132465],
       ['css', 129107],
       ['linux', 127606],
       ['ruby-on-rails', 116883],
       ['windows', 98100],
       ['c', 95453],
       ['sql-server', 74921],
       ['ruby', 73502],
       ['wpf', 65836],
       ['xml', 64157]], dtype=object)

In [45]:
part_number = 2
display(parts[part_number - 1])

array([['ajax', 62239],
       ['database', 59799],
       ['regex', 59223],
       ['windows-7', 58487],
       ['asp.net-mvc', 57859],
       ['xcode', 52513],
       ['django', 52030],
       ['osx', 51812],
       ['arrays', 50055],
       ['vb.net', 46653],
       ['eclipse', 44092],
       ['json', 43451],
       ['facebook', 43393],
       ['ruby-on-rails-3', 43166],
       ['ubuntu', 43002],
       ['performance', 39377],
       ['networking', 38323],
       ['string', 37802],
       ['multithreading', 37619],
       ['winforms', 37370],
       ['security', 34499],
       ['visual-studio-2010', 34433],
       ['asp.net-mvc-3', 34422],
       ['bash', 33116],
       ['homework', 32535]], dtype=object)

In [46]:
part_number = 3
display(parts[part_number - 1])

array([['image', 32280],
       ['wcf', 31490],
       ['html5', 31243],
       ['wordpress', 30556],
       ['visual-studio', 30148],
       ['web-services', 30141],
       ['forms', 30026],
       ['algorithm', 29773],
       ['sql-server-2008', 29743],
       ['linq', 29543],
       ['oracle', 29382],
       ['git', 29377],
       ['query', 28931],
       ['perl', 28641],
       ['apache2', 27619],
       ['flash', 27458],
       ['actionscript-3', 27111],
       ['ipad', 27098],
       ['spring', 26995],
       ['apache', 26990],
       ['silverlight', 26932],
       ['email', 26888],
       ['r', 26863],
       ['cocoa-touch', 26314],
       ['cocoa', 26062]], dtype=object)

In [47]:
part_number = 4
display(parts[part_number - 1])

array([['swing', 25872],
       ['hibernate', 24880],
       ['excel', 24025],
       ['entity-framework', 23978],
       ['file', 23406],
       ['shell', 22309],
       ['flex', 22248],
       ['api', 22037],
       ['list', 21628],
       ['internet-explorer', 21583],
       ['firefox', 21427],
       ['jquery-ui', 21395],
       ['delphi', 21367],
       ['.htaccess', 21022],
       ['sqlite', 20741],
       ['qt', 20586],
       ['tsql', 20544],
       ['google-chrome', 20408],
       ['node.js', 20280],
       ['unix', 20205],
       ['windows-xp', 20193],
       ['http', 19768],
       ['svn', 19727],
       ['unit-testing', 19166],
       ['oop', 19016]], dtype=object)

In [48]:
part_number = 5
display(parts[part_number - 1])

array([['debugging', 18936],
       ['sql-server-2005', 18812],
       ['iis', 18552],
       ['google-app-engine', 18505],
       ['postgresql', 18471],
       ['class', 18363],
       ['codeigniter', 18259],
       ['matlab', 18149],
       ['function', 17967],
       ['ssh', 17866],
       ['validation', 17713],
       ['sockets', 17598],
       ['command-line', 17475],
       ['parsing', 17471],
       ['memory', 17362],
       ['windows-phone-7', 17245],
       ['jsp', 17012],
       ['search', 16901],
       ['templates', 16900],
       ['winapi', 16422],
       ['windows-server-2008', 16323],
       ['google-maps', 16319],
       ['dns', 16179],
       ['authentication', 15932],
       ['events', 15884]], dtype=object)

In [49]:
part_number = 6
display(parts[part_number - 1])

array([['sharepoint', 15701],
       ['calculus', 15679],
       ['xaml', 15665],
       ['pdf', 15619],
       ['zend-framework', 15609],
       ['plugins', 15560],
       ['uitableview', 15544],
       ['visual-studio-2008', 15500],
       ['mongodb', 15499],
       ['linear-algebra', 15457],
       ['scala', 15343],
       ['mvc', 15325],
       ['url', 15163],
       ['magento', 15082],
       ['audio', 14966],
       ['nhibernate', 14863],
       ['real-analysis', 14853],
       ['tomcat', 14848],
       ['design', 14833],
       ['vim', 14790],
       ['session', 14695],
       ['jsf', 14650],
       ['design-patterns', 14634],
       ['android-layout', 14609],
       ['vba', 14490]], dtype=object)

In [50]:
part_number = 7
display(parts[part_number - 1])

array([['google', 14480],
       ['rest', 14426],
       ['optimization', 14304],
       ['video', 14300],
       ['c#-4.0', 14168],
       ['jquery-ajax', 14113],
       ['cakephp', 13882],
       ['variables', 13871],
       ['logging', 13849],
       ['facebook-graph-api', 13830],
       ['testing', 13805],
       ['listview', 13760],
       ['visual-c++', 13740],
       ['table', 13730],
       ['date', 13697],
       ['java-ee', 13579],
       ['ssl', 13545],
       ['sorting', 13541],
       ['centos', 13523],
       ['powershell', 13518],
       ['exception', 13366],
       ['permissions', 13341],
       ['redirect', 13338],
       ['css3', 13332],
       ['gui', 13275]], dtype=object)

In [51]:
part_number = 8
display(parts[part_number - 1])

array([['mod-rewrite', 13275],
       ['gwt', 13235],
       ['caching', 12960],
       ['probability', 12930],
       ['generics', 12901],
       ['drupal', 12895],
       ['math', 12826],
       ['xslt', 12785],
       ['ms-access', 12743],
       ['debian', 12570],
       ['maven', 12482],
       ['active-directory', 12460],
       ['datetime', 12427],
       ['dom', 12360],
       ['web-applications', 12147],
       ['ios5', 12090],
       ['nginx', 11889],
       ['opengl', 11813],
       ['object', 11765],
       ['database-design', 11759],
       ['browser', 11742],
       ['mac', 11698],
       ['loops', 11494],
       ['encryption', 11417],
       ['linq-to-sql', 11376]], dtype=object)

In [52]:
part_number = 9
display(parts[part_number - 1])

array([['phonegap', 11372],
       ['servlets', 11350],
       ['windows-server-2003', 11317],
       ['grails', 11313],
       ['pointers', 11310],
       ['backup', 11309],
       ['haskell', 11298],
       ['animation', 11277],
       ['core-data', 11264],
       ['graphics', 11198],
       ['mobile', 11147],
       ['inheritance', 11135],
       ['configuration', 10961],
       ['abstract-algebra', 10952],
       ['windows-8', 10949],
       ['jquery-mobile', 10909],
       ['iis7', 10789],
       ['gcc', 10776],
       ['post', 10752],
       ['layout', 10742],
       ['select', 10723],
       ['deployment', 10660],
       ['activerecord', 10656],
       ['button', 10644],
       ['geometry', 10586]], dtype=object)

In [53]:
part_number = 10
display(parts[part_number - 1])

array([['application', 10553],
       ['emacs', 10469],
       ['div', 10464],
       ['memory-management', 10241],
       ['fonts', 10228],
       ['jpa', 10139],
       ['data-binding', 10134],
       ['serialization', 10067],
       ['iframe', 10026],
       ['stored-procedures', 9874],
       ['gridview', 9870],
       ['image-processing', 9843],
       ['login', 9802],
       ['spring-mvc', 9708],
       ['opencv', 9674],
       ['version-control', 9651],
       ['cookies', 9616],
       ['text', 9575],
       ['printing', 9541],
       ['dynamic', 9529],
       ['jquery-plugins', 9450],
       ['file-upload', 9409],
       ['amazon-ec2', 9365],
       ['asp.net-mvc-2', 9362],
       ['join', 9289]], dtype=object)

In [54]:
part_number = 11
display(parts[part_number - 1])

array([['boost', 9275],
       ['soap', 9261],
       ['installation', 9222],
       ['reflection', 9197],
       ['architecture', 9196],
       ['scripting', 9180],
       ['proxy', 9179],
       ['netbeans', 9151],
       ['user-interface', 9124],
       ['hard-drive', 9083],
       ['extjs', 9080],
       ['dll', 9028],
       ['blackberry', 9022],
       ['curl', 8986],
       ['ftp', 8966],
       ['script', 8892],
       ['csv', 8702],
       ['combinatorics', 8662],
       ['macros', 8626],
       ['assembly', 8580],
       ['filesystems', 8573],
       ['xpath', 8544],
       ['data-structures', 8519],
       ['javascript-events', 8486],
       ['statistics', 8470]], dtype=object)

In [55]:
part_number = 12
display(parts[part_number - 1])

array([['razor', 8413],
       ['unicode', 8383],
       ['mvvm', 8336],
       ['data', 8253],
       ['android-intent', 8238],
       ['twitter', 8189],
       ['iphone-sdk-4.0', 8177],
       ['windows-vista', 8175],
       ['encoding', 8167],
       ['service', 8140],
       ['usb', 8028],
       ['process', 8001],
       ['general-topology', 7996],
       ['terminal', 7963],
       ['jdbc', 7961],
       ['time', 7925],
       ['canvas', 7922],
       ['tcp', 7911],
       ['azure', 7821],
       ['wireless-networking', 7817],
       ['keyboard', 7799],
       ['routing', 7780],
       ['binding', 7755],
       ['google-maps-api-3', 7754],
       ['file-io', 7696]], dtype=object)

In [56]:
part_number = 13
display(parts[part_number - 1])

array([['backbone.js', 7679],
       ['webserver', 7612],
       ['ant', 7593],
       ['analysis', 7588],
       ['sequences-and-series', 7556],
       ['syntax', 7498],
       ['web', 7491],
       ['view', 7463],
       ['asynchronous', 7459],
       ['drop-down-menu', 7427],
       ['random', 7405],
       ['batch', 7387],
       ['tikz-pgf', 7378],
       ['symfony2', 7367],
       ['collections', 7355],
       ['orm', 7341],
       ['https', 7332],
       ['selenium', 7302],
       ['group-theory', 7272],
       ['uiview', 7259],
       ['if-statement', 7258],
       ['graph', 7244],
       ['opengl-es', 7243],
       ['actionscript', 7238],
       ['language-agnostic', 7198]], dtype=object)

In [57]:
part_number = 14
display(parts[part_number - 1])

array([['vpn', 7178],
       ['excel-vba', 7176],
       ['algebra-precalculus', 7144],
       ['heroku', 7137],
       ['recursion', 7118],
       ['input', 7091],
       ['reporting-services', 7045],
       ['matrices', 7036],
       ['windows-server-2008-r2', 7024],
       ['.net-4.0', 7010],
       ['number-theory', 6969],
       ['compiler', 6958],
       ['complex-analysis', 6957],
       ['joomla', 6926],
       ['methods', 6903],
       ['virtualization', 6897],
       ['types', 6883],
       ['entity-framework-4', 6854],
       ['twitter-bootstrap', 6828],
       ['uitableviewcell', 6771],
       ['hash', 6762],
       ['properties', 6762],
       ['formatting', 6647],
       ['website', 6610],
       ['concurrency', 6587]], dtype=object)

In [58]:
part_number = 15
display(parts[part_number - 1])

array([['com', 6586],
       ['colors', 6573],
       ['url-rewriting', 6573],
       ['asp.net-mvc-4', 6558],
       ['sharepoint2010', 6549],
       ['open-source', 6511],
       ['amazon-web-services', 6485],
       ['2010', 6457],
       ['boot', 6431],
       ['dictionary', 6404],
       ['interface', 6398],
       ['utf-8', 6381],
       ['frameworks', 6376],
       ['memory-leaks', 6369],
       ['datagrid', 6367],
       ['router', 6358],
       ['ip', 6347],
       ['django-models', 6338],
       ['for-loop', 6337],
       ['virtualbox', 6332],
       ['exception-handling', 6312],
       ['groovy', 6287],
       ['microsoft-excel', 6284],
       ['map', 6263],
       ['cron', 6244]], dtype=object)

In [59]:
part_number = 16
display(parts[part_number - 1])

array([['asp-classic', 6242],
       ['batch-file', 6232],
       ['vector', 6231],
       ['jsf-2', 6221],
       ['internet-explorer-8', 6167],
       ['coldfusion', 6143],
       ['smtp', 6118],
       ['mercurial', 6103],
       ['import', 6092],
       ['stl', 6070],
       ['reference-request', 6064],
       ['activity', 6055],
       ['xcode4', 6052],
       ['autocomplete', 6049],
       ['upload', 6048],
       ['functional-analysis', 6044],
       ['parameters', 6022],
       ['character-encoding', 5983],
       ['programming-languages', 5976],
       ['path', 5960],
       ['build', 5946],
       ['datagridview', 5940],
       ['vbscript', 5932],
       ['3d', 5928],
       ['localization', 5919]], dtype=object)

In [4]:
import pandas as pd

se_tags = {
    "java", "c#", "javascript", "c++", "python", "android", "ios", "sql",
    "git", "oop", "multithreading", "design-patterns", "architecture",
    "database-design", "unit-testing", "version-control", "api",
    "web-services", "debugging", "mvc"
}

net_tags = {
    "networking", "linux", "ubuntu", "security", "bash", "apache", "http",
    "ssh", "sockets", "dns", "ssl", "centos", "nginx", "tcp", "routing",
    "wireless-networking", "ftp", "proxy", "active-directory", "amazon-ec2"
}

ai_tags = {
    "opencv", "image-processing", "r", "matlab", "algorithm", "statistics",
    "probability", "linear-algebra", "optimization", "matrices", "vector",
    "graph", "data-structures", "search"
}







In [4]:
def has_any_tag(tags_text, target_tags):
    tags = str(tags_text).split()
    return any(tag in target_tags for tag in tags)

se_list = []
net_list = []
ai_list = []

list of data frame

In [5]:
for chunk in pd.read_csv("Train.csv", chunksize=50000):
    se_list.append(chunk[chunk["Tags"].apply(lambda x: has_any_tag(x, se_tags))])
    net_list.append(chunk[chunk["Tags"].apply(lambda x: has_any_tag(x, net_tags))])
    ai_list.append(chunk[chunk["Tags"].apply(lambda x: has_any_tag(x, ai_tags))])

train_se = pd.concat(se_list, ignore_index=True)
train_net = pd.concat(net_list, ignore_index=True)
train_ai = pd.concat(ai_list, ignore_index=True)

train_se.to_csv("train_se.csv", index=False)
train_net.to_csv("train_net.csv", index=False)
train_ai.to_csv("train_ai.csv", index=False)

In [69]:
print("SE:", len(train_se))
print("NET:", len(train_net))
print("AI:", len(train_ai))

SE: 2256087
NET: 404428
AI: 173455


In [70]:

se_df = pd.read_csv("train_se.csv")
net_df = pd.read_csv("train_net.csv")
ai_df = pd.read_csv("train_ai.csv")

print("=== SE SAMPLE ===")
display(se_df[["Title", "Tags"]].sample(10, random_state=42))

print("=== NET SAMPLE ===")
display(net_df[["Title", "Tags"]].sample(10, random_state=42))

print("=== AI SAMPLE ===")
display(ai_df[["Title", "Tags"]].sample(10, random_state=42))

=== SE SAMPLE ===


,Title,Tags
1755038,Ident connection fails via psycopg2 but works via command line,python postgresql authentication psycopg2
1602316,"Python List - ""reserving"" space ( ~ resizing)",python python-3.x
2142174,Java lib to build and print table on console?,java table
1672221,Char[] to Byte[] for output optimize in web (java),java performance bytearray velocity char-array
1525431,How to save a row in another table after a period of time in a Windows service,c# sql-server-2005
427994,chicken and egg Spring bean binding,java spring inversion-of-control
2088510,How can I select data with linq to sql from another query?,sql asp.net-mvc linq-to-sql
439876,Android Mime Type for certificates.,android android-intent
758582,Regex match base64 encoding in email subject,java regex
717715,Not showing the images in live server which is showing in my local host,c# asp.net


=== NET SAMPLE ===


,Title,Tags
93013,Copy and paste not working under rdesktop,linux ubuntu-10.04 windows-server-2008 rdesktop
239468,Flash player problems in Chrome on Linux Mint 12,linux google-chrome linux-mint flash-player adobe-flash
191698,How to set the network profile of Windows 7 via group policy?,networking windows-7 windows-domain
166104,How to create a software raid5 array without a spare,linux software-raid mdadm
345940,How to start plasma-desktop from SSH-console on desktop session?,linux ssh opensuse kde tty
299954,Using cURL to get google chart images through a proxy,php curl proxy google-charts
198271,twitter with curl proxy,twitter curl proxy
220952,Prototyping Web App Framework,web-applications frameworks centos
251510,CentOS 6 - iptables preventing web access via port 80,linux apache centos iptables
360625,How do I check which process is using port 1099?,networking process port


=== AI SAMPLE ===


,Title,Tags
100646,Munin Graphs - Have one graph/plugin only update once per hour?,monitoring performance-monitoring munin graph
51050,Determinant of block matrix,linear-algebra determinant
69534,Why aren't all real self-adjoint operators diagonal?,linear-algebra
35152,What's inside a haar cascade classifier in Open CV computer vision?,xml opencv classifier
146203,Traversing weighted graph through all vertecies ending up at the same point,algorithm graph traversal
96216,Advanced statistics tracking with nginx,php nginx statistics request tracking
86753,"Ways to do ""related searches"" functionality",search information-retrieval
71768,How can I optimize this simple PHP script?,php optimization file performance
34548,R: Change the fields' (slots') values of the class assigning a value to some other field,r
27512,How can I derive a variable in R showing the number of observations that have the same value recorded at earlier dates?,r


In [6]:

ai_tags = {
    "opencv",
    "image-processing",
    "r",
    "matlab",
    "algorithm",
    "probability",
    "linear-algebra",
    "matrices",
    "vector",
    "data-structures"
}

def has_any_tag(tags_text, target_tags):
    tags = str(tags_text).split()
    return any(tag in target_tags for tag in tags)

ai_list = []

for chunk in pd.read_csv("Train.csv", chunksize=50000):
    ai_list.append(chunk[chunk["Tags"].apply(lambda x: has_any_tag(x, ai_tags))])

train_ai = pd.concat(ai_list, ignore_index=True)
train_ai.to_csv("train_ai_1.csv", index=False)

print("AI clean:", len(train_ai))

AI clean: 133850


In [73]:
import pandas as pd

ai_df = pd.read_csv("train_ai_1.csv")
display(ai_df[["Title", "Tags"]].sample(20, random_state=42))

,Title,Tags
32761,Looping a function for each row in csv,r for-loop
115561,Creating a data frame from two vectors using cbind,r data.frame
133429,How to sort/draw pseudo-3D buildings so as they don't visually overlap,algorithm map drawing
8835,"A2R library colored dendrogram, allow more than six chars per label",r
70485,Algorithm to find the smallest snippet from searching a document?,algorithm
73840,Detecting image differences,image-processing
4988,Minimal polynomial and Hermitian matrix,linear-algebra matrices
111348,I have an OpenGL Tessellated Sphere and I want to cut a cylindrical hole in it,algorithm opengl graphics tesselation
87072,Cell array within nested loop,matlab
99609,How do I get many frames at once?,python opencv mpi


In [3]:
# الكلمات المشبوهة التي نريد استبعاد الصفوف التي تحتويها
bad_keywords = [
    "<?php",
    "shell_exec",
    "passthru(",
    "base64_decode",
    "eval(",
    "system(",
    "exec(",
    "cmd="
]

def has_any_tag(tags_text, target_tags):
    tags = str(tags_text).split()
    return any(tag in target_tags for tag in tags)

def looks_suspicious(text):
    text = str(text).lower()
    return any(k.lower() in text for k in bad_keywords)

In [5]:
# 1) تنظيف SE مباشرة من Train.csv باستخدام se_tags
chunks = []

for chunk in pd.read_csv("Train.csv", chunksize=50000):
    se_chunk = chunk[chunk["Tags"].apply(lambda x: has_any_tag(x, se_tags))].copy()
    if "Body" in se_chunk.columns:
        se_chunk = se_chunk[~se_chunk["Body"].apply(looks_suspicious)]
    chunks.append(se_chunk)

train_se_clean = pd.concat(chunks, ignore_index=True)
train_se_clean.to_csv("train_se_clean.csv", index=False)
print("train_se_clean.csv", len(train_se_clean))



train_se_clean.csv 2236180


In [6]:
for input_file, output_file in [
    ("train_net.csv", "train_net_clean.csv"),
    ("train_ai_1.csv", "train_ai_clean.csv"),
]:
    print("\nProcessing:", input_file)

    chunks = []

    for chunk in pd.read_csv(input_file, chunksize=50000):
        print("Columns:", chunk.columns)

        if "Body" in chunk.columns:
            before = len(chunk)
            chunk = chunk[~chunk["Body"].apply(looks_suspicious)]
            after = len(chunk)

            print(f"Filtered: {before} -> {after}")

            chunks.append(chunk)
        else:
            print("❌ No Body column!")

    if len(chunks) == 0:
        print("⚠️ No data collected!")
        continue

    clean_df = pd.concat(chunks, ignore_index=True)
    clean_df.to_csv(output_file, index=False)

    print("Saved:", output_file, len(clean_df))


Processing: train_net.csv
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49521
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49528
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49519
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49524
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49540
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49514
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49548
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49547
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 4428 -> 4380
Saved: train_net_clean.csv 400621

Processing: train_ai_1.csv
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Filtered: 50000 -> 49604
Columns: Index(['Id', 'Title', 'Body', 'Tags'], dtype